In [1]:
import os

# Force TensorFlow to use CPU to avoid CUDA init errors during Hailo SDK import.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

'2'

In [2]:
from pathlib import Path

import onnx

from hailo_sdk_client import ClientRunner

E0000 00:00:1778947174.700849   29062 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778947174.716404   29062 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0516 17:59:53.835000 29062 torch/utils/cpp_extension.py:117] No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


### Parsing ONNX to Hailo

Number of classes:
- cifar10: 10
- cifar100: 100
- flowers: 102
- food: 101
- pets: 37

In [3]:
chosen_hw_arch = 'hailo8'

# Create a classifier instance
device = "cpu"
batch_size = 8
num_classes = 10
channels = 3
img_h, img_w = 224, 224

optimization_level = 0  # -100, 0-4
compression_level = 0   # 0-5

model_name = 'resnet10t' 
model_frozen_layer_num = {
    'resnet18': 12,
    'resnet34': 20,
    'mobilenetv2_100': 21,
    'efficientnet_b2': 27,
    'visformer_tiny': 24,
    'lcnet_100': 19,
    'semnasnet_100': 20,
    'mobilenetv3_large_100': 22,
    'resnet10t': 14,
}
frozen_layer_num = model_frozen_layer_num[model_name]

artifacts_path = Path(f'../artifacts/{model_name}/frozen_layer_{frozen_layer_num}_classes_{num_classes}')

In [4]:
runner = ClientRunner(hw_arch=chosen_hw_arch)

model_path = artifacts_path / f'{model_name}.onnx'
start_node_names = ['input']
model_end_node_names = {
    'resnet18': ['/frozen_features/frozen_features.12/flatten/Flatten'],
    'resnet34': ['/frozen_features/frozen_features.20/flatten/Flatten'],
    'mobilenetv2_100': ['/frozen_features/frozen_features.21/flatten/Flatten'],
    'efficientnet_b2': ['/frozen_features/frozen_features.27/flatten/Flatten'],
    'visformer_tiny': [''],
    'lcnet_100': ['/frozen_features/frozen_features.19/Flatten'],
    'semnasnet_100': ['/frozen_features/frozen_features.20/flatten/Flatten'],
    'mobilenetv3_large_100': ['/frozen_features/frozen_features.22/Flatten'],
    'resnet10t': ['/frozen_features/frozen_features.14/flatten/Flatten']
}
end_node_names = model_end_node_names[model_name]
net_input_shapes = {'input': [1, channels, img_h, img_w],}

In [5]:
hn, npz = runner.translate_onnx_model(
    str(model_path),
    model_name,
    start_node_names=start_node_names,
    end_node_names=end_node_names,
    net_input_shapes=net_input_shapes,
)

[info] Translation started on ONNX model resnet10t
[info] Restored ONNX model resnet10t (completion time: 00:00:00.12)
[info] Extracted ONNXRuntime meta-data for Hailo model (completion time: 00:00:00.52)
[info] Start nodes mapped from original model: 'input': 'resnet10t/input_layer1'.
[info] End nodes mapped from original model: '/frozen_features/frozen_features.14/flatten/Flatten'.
[info] Translation completed on ONNX model resnet10t (completion time: 00:00:00.73)


In [6]:
hailo_har_model_path = artifacts_path / f'hailo8_optim_{optimization_level}_comp_{compression_level}' / f'{model_name}.har'
hailo_har_model_path.parent.mkdir(parents=True, exist_ok=True)
runner.save_har(hailo_har_model_path)

[info] Saved HAR to: /home/mpiechocki/research/accelerator-training/artifacts/resnet10t/frozen_layer_14_classes_10/hailo8_optim_0_comp_0/resnet10t.har


In [7]:
# !hailo profiler {hailo_har_model_path}

#### Model optimization

In [8]:
import numpy as np
from tqdm import tqdm
from PIL import Image


images_list = list(Path('../data/ILSVRC2012_img_val/').rglob('*.JPEG'))

img_mean = [123.675, 116.28, 103.53]
img_std = [58.395, 57.12, 57.375]

# calib_dataset = np.random.randint(0, 255, (1024, img_h, img_w, channels))
calib_dataset = np.random.random((1024, img_h, img_w, channels))

## calib_dataset = np.zeros((len(images_list), img_h, img_w, channels))
# calib_set_size = 1024
# calib_dataset = np.zeros((calib_set_size, img_h, img_w, channels))
# for idx, img_path in enumerate(tqdm(sorted(images_list[:calib_set_size]))):
#     img = np.array(Image.open(img_path).convert('RGB').resize((img_h, img_w)))
#     img = (img.astype(np.float32) - img_mean) / img_std
#     # img = np.transpose(img, (2, 0, 1))
#     img = np.expand_dims(img, axis=0).astype(np.float32)
#     calib_dataset[idx, :, :, :] = img

# calib_set_path = Path('./calib_set.npy')
# np.save(calib_set_path, calib_dataset)

# calib_dataset = np.load(calib_set_path)
print(f'Calibration dataset shape: {calib_dataset.shape}')
print(f'Calibration dataset size: {calib_dataset.nbytes / 1024**2:.2f} MB')

Calibration dataset shape: (1024, 224, 224, 3)
Calibration dataset size: 1176.00 MB


In [ ]:
# Call Optimize to perform the optimization process
runner = ClientRunner(hw_arch=chosen_hw_arch, har=str(hailo_har_model_path))

# Now we will create a model script, that tells the compiler to add a normalization on the beginning
# of the model (that is why we didn't normalize the calibration set;
# Otherwise we would have to normalize it before using it)

# Batch size is 8 by default
# alls = 'normalization1 = normalization([123.675, 116.28, 103.53], [58.395, 57.12, 57.375])\n'

model_script_commands = [
    f'model_optimization_flavor(optimization_level={optimization_level}, compression_level={compression_level}, batch_size=8)\n'
]

# Load the model script to ClientRunner so it will be considered on optimization
runner.load_model_script(''.join(model_script_commands))

runner.optimize(calib_dataset)

# Save the result state to a Quantized HAR file
quantized_har_model_path = str(hailo_har_model_path).replace('.har', '_quantized_model.har')
runner.save_har(quantized_har_model_path)

[info] Loading model script commands to resnet10t from string
[info] Found model with 3 input channels, using real RGB images for calibration instead of sampling random data.
[info] Starting Model Optimization


In [ ]:
# runner.analyze_noise(calib_dataset, data_count=16, batch_size=2)

In [ ]:
# !hailo profiler {quantized_har_model_path}

#### Quantized model compilation to HEF format

In [ ]:
runner = ClientRunner(har=quantized_har_model_path)
hef = runner.compile()

hef_model_path = quantized_har_model_path.replace('har', 'hef')
with open(hef_model_path, 'wb') as f:
    f.write(hef)

[info] To achieve optimal performance, set the compiler_optimization_level to "max" by adding performance_param(compiler_optimization_level=max) to the model script. Note that this may increase compilation time.
[info] Loading network parameters
[info] Starting Hailo allocation and compilation flow
[info] Building optimization options for network layers...
[info] Successfully built optimization options - 1s 363ms
[info] Trying to compile the network in a single context
[info] Using Single-context flow
[info] Resources optimization params: max_control_utilization=75%, max_compute_utilization=75%, max_compute_16bit_utilization=75%, max_memory_utilization (weights)=75%, max_input_aligner_utilization=75%, max_apu_utilization=75%
[info] Validating layers feasibility
[info] output_layer1: Pass
[info] input_layer1: Pass
[info] conv1_sd1: Pass
[info] smuffers_shortcut_conv2_to_conv3: Pass
[info] conv3_sd2: Pass
[info] conv2_sd1: Pass
[info] conv1_sd0: Pass
[info] conv1_sdc: Pass
[info] sh_from

#### Export to ONNX with HailoOp operator

In [ ]:
onnx_model = runner.get_hailo_runtime_model()
onnx_artifacts_path = hef_model_path.replace('.hef', '_hailo.onnx')
onnx_file = onnx.save(onnx_model, onnx_artifacts_path)
print(f'ONNX with HailoOp saved as: {onnx_artifacts_path}')

[warning] GlobalAvgPool output shape might be different from ONNX shape
ONNX with HailoOp saved as: ../artifacts/resnet10t/frozen_layer_14_classes_102/hailo8_optim_2_comp_0/resnet10t_quantized_model_hailo.onnx
